# Prompting Tejas — From GMA Desired Future Conditions to MODFLOW Parameters

DSO Summer Institute 2026 · Day 2 (Afternoon)

This notebook is the hands-on companion to *Prompting Tejas*. It uses the helpers in `utils.py` to talk to TACC's air-gapped Tejas LLM service through an OpenAI-compatible API.

You will run three exercises:

1. **Exercise A** — translate a Desired Future Condition (DFC) statement into MODFLOW input parameters using a tagged `<context>/<task>/<format>` prompt.
2. **Exercise B** — chain three prompts: extract entities → extract candidate SVOs (noun + unit-of-measure) → extract decision-relevant claims with confidence levels.
3. **Exercise C** — load a real GMA / DFC document (PDF or text) and re-run the mapping on it.

A self-scoring rubric (DFC understanding × MODFLOW mapping × hydrogeological realism) is included at the end so you can grade each output before iterating.

## 1. Import the helpers

In [1]:
import os
from dataclasses import replace

from utils import LLMConfig, run_llm, load_context, SYSTEM_PROMPT, USER_PROMPT_TEMPLATE

print("System prompt:\n", SYSTEM_PROMPT)

System prompt:
 You are a helpful assistant supporting researchers in the DSO Institute. Answer concisely and cite uncertainty when it exists.


## 1b. Configure the model

The `semantic_bridge` package expects `model`, `api_key`, and `base_url` to be passed explicitly to every LLM call. We mirror that pattern here: read them from environment variables once, then reuse the same `LLMConfig` across the notebook. Any later cell can still override `model=`, `api_key=`, or `base_url=` per call.

This loads the same `.env` file used by the Morning notebooks (`../Morning/.env`), so you don't need to duplicate keys. The relevant variables are:

- `OPENAI_API_KEY` — required
- `OPENAI_BASE_URL` — TACC's TEJAS endpoint (`https://ai.tejas.tacc.utexas.edu`) for the workshop; leave unset for the default OpenAI endpoint
- `TOPIC_LABELER_MODEL` — model name (e.g. `Meta-Llama-3.3-70B-Instruct`); falls back to `OPENAI_MODEL` then the `utils.DEFAULT_MODEL`

In [2]:
from pathlib import Path

from dotenv import load_dotenv

# Share configuration with the Morning notebooks.
morning_env = Path("../Morning/.env").resolve()


TEJAS_API_KEY = "sk-Cv7V8lAPD5bEZPEYNUlSAQ"
TEJAS_BASE_URL = "https://ai.tejas.tacc.utexas.edu"
TEJAS_MODEL = "Llama-4-Maverick-17B-128E-Instruct"
# "Meta-Llama-3.3-70B-Instruct"
# "E5-Mistral-7B-Instruct"


base_config = LLMConfig(
    model=TEJAS_MODEL,
    api_key=TEJAS_API_KEY,
    base_url=TEJAS_BASE_URL,
    # Long structured-extraction outputs (entities, SVOs, claims) need more
    # head-room than the 800-token default, otherwise JSON mode truncates.
    max_tokens=4096,
)

print(f"model    = {base_config.model}")
print(f"base_url = {base_config.base_url or '<default>'}")
print(f"api_key  = {'set' if base_config.api_key else 'missing'}")
print(f"max_tokens = {base_config.max_tokens}")

model    = Llama-4-Maverick-17B-128E-Instruct
base_url = https://ai.tejas.tacc.utexas.edu
api_key  = set
max_tokens = 4096


## 2. The three-part prompt template

Every call below uses the same tagged structure (slide 7 of the deck). Tejas listens for these tags, and they make rubric scoring deterministic.

```
<context> background a hydrogeologist would need on this problem  </context>
<task>    what you want the model to do, step by step              </task>
<format>  the shape of the answer (JSON schema, table, bullets…)   </format>
```

`USER_PROMPT_TEMPLATE` in `utils.py` already wraps your inputs in those tags, so you just pass `context=`, `task=`, and `format=` keyword arguments.

**Four ground rules** (slide 4):

1. Be specific — say what you want, not what you don't.
2. Structure input and output with tags / delimiters.
3. Split complex tasks — one job per prompt, chain the outputs.
4. Show, don't just tell — one example beats three sentences of description.

In [3]:
print(USER_PROMPT_TEMPLATE)

<context>
{context}
</context>

<task>
{task}
</task>

<format>
{format}
</format>


## 3. Exercise A — GMA DFC → MODFLOW parameters

> "GMA 12 sets a 50-yr average drawdown DFC of ≤ 25 ft for the Carrizo–Wilcox. Which MODFLOW inputs — pumping (WEL), recharge (RCH), conductivity (K), storage (S) — determine whether that target is reachable?"

We ask Tejas to (1) extract each testable condition from the DFC and (2) map it to MODFLOW packages and parameters. The `<format>` block forces structured JSON so the output is easy to evaluate.

In [4]:
import json

dfc_context = """
Groundwater Management Area: GMA 12
Aquifer: Carrizo-Wilcox
Planning horizon: 50 years
DFC statement: "The 50-year average drawdown in the Carrizo-Wilcox aquifer
shall not exceed 25 feet, measured across representative monitoring wells
within GMA 12."
Candidate MODFLOW packages of interest: WEL (pumping), RCH (recharge),
NPF/LPF (hydraulic conductivity K), STO (specific storage / yield S).
""".strip()

dfc_task = """
1. Extract each testable condition implied by the DFC (target value, units, horizon, spatial scope).
2. For each condition, identify which MODFLOW input package and parameter governs whether the
   target is reachable.
3. Note the expected sensitivity of the condition to that parameter (high / medium / low).
4. List one source of uncertainty a hydrogeologist would flag for each mapping.
""".strip()

dfc_format = """
Return a JSON object with a single key "mappings" whose value is a list of objects with fields:
- condition         (string, the testable condition extracted from the DFC)
- modflow_package   (string, e.g. "WEL", "RCH", "NPF", "STO")
- parameter         (string, e.g. "pumping rate", "recharge flux", "Kx", "Ss")
- sensitivity       (one of "high", "medium", "low")
- uncertainty_notes (string, one sentence)
Reply with valid JSON only, no prose, no markdown.
""".strip()

dfc_mapper = replace(
    base_config,
    system_prompt=(
        "You are a hydrogeologist assisting Texas Water Development Board planners. "
        "Translate Desired Future Condition statements into MODFLOW input parameters. "
        "Be precise, cite uncertainty, and return strictly valid JSON when asked."
    ),
    temperature=0.1,
    extra={"response_format": {"type": "json_object"}},
)

raw = run_llm(dfc_mapper, context=dfc_context, task=dfc_task, format=dfc_format)
mappings = json.loads(raw)
print(json.dumps(mappings, indent=2))

{
  "mappings": [
    {
      "condition": "50-year average drawdown not exceeding 25 feet",
      "modflow_package": "WEL",
      "parameter": "pumping rate",
      "sensitivity": "high",
      "uncertainty_notes": "Uncertainty in future pumping rates due to changes in water demand or policy."
    },
    {
      "condition": "50-year average drawdown not exceeding 25 feet",
      "modflow_package": "RCH",
      "parameter": "recharge flux",
      "sensitivity": "medium",
      "uncertainty_notes": "Uncertainty in future recharge rates due to climate change or land use changes."
    },
    {
      "condition": "50-year average drawdown not exceeding 25 feet",
      "modflow_package": "NPF",
      "parameter": "Kx",
      "sensitivity": "high",
      "uncertainty_notes": "Uncertainty in hydraulic conductivity due to heterogeneity and limited data."
    },
    {
      "condition": "50-year average drawdown not exceeding 25 feet",
      "modflow_package": "STO",
      "parameter": "Ss",
 

## 4. Load the source document

Exercises B and C both run against a real document. `load_context()` reads a `.txt`, `.md`, or `.pdf` file into a string; for PDFs it uses `pypdf`.

Swap `PDF_URL` for the report or DFC document your team is studying. The cell falls back to a local `sample_context.txt` if the download fails (e.g. when you're off the TACC network).

In [5]:
import urllib.request
from pathlib import Path

# Replace with the URL of the GMA / DFC PDF you want to analyze.
PDF_URL = "https://ckan.tacc.utexas.edu/dataset/06fc7fdc-ba68-499e-b8ef-f1f0aa136544/resource/bde10ad9-412c-4f94-801d-b74dbc43da9e/download/1961.pdf"

downloads_dir = Path("downloads")
downloads_dir.mkdir(exist_ok=True)
pdf_path = downloads_dir / Path(PDF_URL).name

if not pdf_path.exists():
    print(f"Downloading {PDF_URL} ...")
    try:
        urllib.request.urlretrieve(PDF_URL, pdf_path)
    except Exception as exc:
        # Fall back to the synthetic Exercise A context if the URL is unreachable.
        print(f"Download failed ({exc}); falling back to sample_context.txt")
        pdf_path = Path("sample_context.txt")
        if not pdf_path.exists():
            pdf_path.write_text(dfc_context)

document_text = load_context(pdf_path, max_chars=8000)
print(f"Loaded {len(document_text)} characters from {pdf_path.name}\n")
print(document_text[:500], "...")

Loaded 8015 characters from 1961.pdf

TEXAS BOARD OF WATER ENGINEERS 
Durwood Manford, Chairman 
R. M. Dixon, Member 
0. F. Dent, Member 
A PLAN FOR MEETING THE 1980 WATER 
REQUIREMENTS OF TEXAS 
Prepared under the direction of 
John J. Vandertulip, Chief Engineer 
For Submittal to the 
Fifty-Seventh Legislature 
May 1961 

BOARD OF WATER ENGINEERS 
DURWOOD MANFORD. CHAIRMAN 
R. M. DIXON 
0. F. DENT 
BEN F. LOONEY. JR. 
SECRETARY 
Honorable Price Daniel 
Governor of Texas 
Austin, Texas 
Honorable Ben Ramsey 
Lieutenant Governor of  ...


## 5. Exercise B — A three-prompt chain over the document

From slide 2 of the deck, the prompt sequence is:

1. **Prompt 1** — extract entities (people, places, datasets, instruments).
2. **Prompt 2** — identify candidate SVOs: noun + unit-of-measure pairs (Stoica & Peckham, 2019).
3. **Prompt 3** — extract decision-relevant claims with confidence levels.

All three prompts run against the `document_text` you loaded in section 5, so the outputs reflect the real document — not a hand-crafted passage. Chaining narrows the model's focus and yields outputs that can feed CKAN registration.

In [6]:
# --- Prompt 1: entity extraction over the loaded document ---
entity_extractor = replace(
    base_config,
    system_prompt="You extract structured entities from hydrogeology text. Reply with strict JSON only.",
    temperature=0.0,
    extra={"response_format": {"type": "json_object"}},
)

entity_task = (
    "Extract every named entity from the document and classify each one as "
    "person, place, dataset, instrument, agency, or aquifer."
)
entity_format = """
JSON object with a single key "entities" whose value is a list of objects with fields:
- text   (string, the surface form as it appears)
- type   (one of: person, place, dataset, instrument, agency, aquifer, other)
- note   (string, one-clause justification)
"""

entities = json.loads(run_llm(
    entity_extractor,
    context=document_text,
    task=entity_task,
    format=entity_format.strip(),
))
print(json.dumps(entities, indent=2))

{
  "entities": [
    {
      "text": "TEXAS BOARD OF WATER ENGINEERS",
      "type": "agency",
      "note": "The entity is referred to as a governing body"
    },
    {
      "text": "Durwood Manford",
      "type": "person",
      "note": "Named as Chairman"
    },
    {
      "text": "R. M. Dixon",
      "type": "person",
      "note": "Named as Member"
    },
    {
      "text": "0. F. Dent",
      "type": "person",
      "note": "Named as Member"
    },
    {
      "text": "John J. Vandertulip",
      "type": "person",
      "note": "Named as Chief Engineer"
    },
    {
      "text": "TEXAS",
      "type": "place",
      "note": "State mentioned in the context"
    },
    {
      "text": "Canadian River Basin",
      "type": "place",
      "note": "Geographical area mentioned"
    },
    {
      "text": "Red River Basin",
      "type": "place",
      "note": "Geographical area mentioned"
    },
    {
      "text": "Sulphur River Basin",
      "type": "place",
      "note": "Geog

### Prompt 2 — Candidate SVOs (noun + unit-of-measure)

An SVO is the (object, variable, unit) triple at the heart of the Scientific Variable Ontology. These are what CKAN can match to existing datasets, so the cleaner this list, the better the downstream linking.

In [10]:
svo_extractor = replace(
    base_config,
    system_prompt="You extract SVO (Scientific Variable Ontology) candidates: object + variable + unit-of-measure.",
    temperature=0.0,
    extra={"response_format": {"type": "json_object"}},
)

svo_task = """
You are an ontology-aware scientific metadata assistant.

Your task is to read the provided document and identify scientific variables that can be represented using the Scientific Variables Ontology (SVO) framework associated with Scott D. Peckham and Maria Stoica.

In SVO, a scientific variable should not be treated as a simple label. It should be decomposed into atomistic components, especially:

1. Phenomenon — the thing, substance, object, system, event, or process being observed.
2. Property — the measurable or observable attribute of that phenomenon.
3. Optional modifiers — context needed to make the variable unambiguous, such as:
   - process
   - medium or material
   - location or spatial context
   - temporal context
   - mathematical operation, such as mean, maximum, minimum, rate, derivative, integral, anomaly, total, concentration, flux, etc.
   - measurement method, instrument, model, or observational context
   - units, if stated
   - aggregation interval, if stated

Your goal is to extract candidate SVO-style variable definitions from the document.

## Instructions

Read the document carefully and identify terms that represent scientific variables, measurements, model inputs, model outputs, observed quantities, derived quantities, or dataset fields.

For each candidate variable:

1. Preserve the original variable phrase exactly as written in the document.
2. Decompose it into SVO-style components.
3. Identify the primary phenomenon.
4. Identify the property.
5. Identify any modifiers.
6. Determine whether the variable is directly stated, inferred from context, or ambiguous.
7. Provide a short explanation of your reasoning.
8. Do not invent information that is not supported by the document.
9. If a component is unclear, use `null` and explain what is missing.
10. If multiple SVO interpretations are possible, include each as a separate candidate with a confidence score.



## Important Rules

- A valid SVO-style variable must include at least a phenomenon and a property.
- Do not treat a dataset name, project name, method name, or place name as a variable unless it clearly refers to a measured or modeled quantity.
- Prefer the most specific phenomenon supported by the text.
- Prefer the most specific property supported by the text.
- If the document says "soil moisture", treat "soil" as the phenomenon and "moisture" or "water content" as the property.
- If the document says "daily average soil moisture", identify:
  - phenomenon: soil
  - property: moisture or water content
  - temporal context: daily
  - mathematical operation: average
- If the document says "streamflow", consider whether it means:
  - phenomenon: stream/channel/river water
  - property: volumetric flow rate/discharge
  Explain the choice.
- If the document says "temperature" without specifying the medium, mark the phenomenon as ambiguous unless context clearly indicates air, water, soil, surface, etc.
- If units are available, use them to clarify the property.
- Separate raw variables from derived variables when possible.
- Return only JSON. Do not include markdown, commentary, or explanation outside the JSON.

## Document to Analyze

<<<DOCUMENT_TEXT_HERE>>>
"""

svo_format = """
## Output Format

Return strict JSON only.

Use this schema:

{
  "document_title": "<title if available, otherwise null>",
  "variables": [
    {
      "original_phrase": "<exact phrase from document>",
      "normalized_label": "<clean human-readable label>",
      "svo_candidate_name": "<phenomenon__property style name, with modifiers if useful>",
      "phenomenon": {
        "primary": "<main observed thing>",
        "secondary": "<related object, medium, domain, or context if applicable>",
        "process": "<process if applicable, otherwise null>"
      },
      "property": "<measured or modeled attribute>",
      "modifiers": {
        "spatial_context": "<location, depth, region, grid cell, station, watershed, etc., otherwise null>",
        "temporal_context": "<daily, monthly, instantaneous, annual, time range, etc., otherwise null>",
        "mathematical_operation": "<mean, maximum, minimum, total, rate, flux, anomaly, etc., otherwise null>",
        "method_or_instrument": "<sensor, model, assay, satellite, field method, etc., otherwise null>",
        "units": "<units if stated, otherwise null>"
      },
      "evidence": {
        "source_text": "<short quote or paraphrase from the document>",
        "section_or_location": "<section, table, figure, page, or null>"
      },
      "status": "direct | inferred | ambiguous",
      "confidence": 0.0,
      "reasoning": "<brief explanation of why this is the phenomenon/property decomposition>",
      "missing_information": [
        "<items needed to make the SVO definition more precise>"
      ]
    }
  ],
  "ambiguous_terms": [
    {
      "term": "<term from document>",
      "issue": "<why it could not be confidently decomposed>",
      "possible_interpretations": [
        {
          "phenomenon": "<possible phenomenon>",
          "property": "<possible property>",
          "confidence": 0.0
        }
      ]
    }
  ],
  "notes": [
    "<any document-level observations about variable naming, units, ambiguity, or metadata quality>"
  ]
}
"""

svos = json.loads(run_llm(
    svo_extractor,
    context=document_text,
    task=svo_task.strip(),
    format=svo_format.strip(),
))
print(json.dumps(svos, indent=2))

{
  "document_title": "A Plan for Meeting the 1980 Water Requirements of Texas",
  "variables": [
    {
      "original_phrase": "water requirements",
      "normalized_label": "Water Requirements",
      "svo_candidate_name": "water__demand",
      "phenomenon": {
        "primary": "water",
        "secondary": null,
        "process": null
      },
      "property": "demand",
      "modifiers": {
        "spatial_context": "Texas",
        "temporal_context": "1980",
        "mathematical_operation": null,
        "method_or_instrument": null,
        "units": null
      },
      "evidence": {
        "source_text": "A Plan for Meeting the 1980 Water Requirements of Texas",
        "section_or_location": "Title"
      },
      "status": "direct",
      "confidence": 0.8,
      "reasoning": "The document title directly mentions 'water requirements', indicating a clear focus on the demand for water.",
      "missing_information": [
        "units"
      ]
    },
    {
      "original_

### Prompt 3 — Decision-relevant claims with confidence

The model now reads the passage as a planner would, surfacing claims that could feed a decision and flagging how confident the language is.

In [8]:
claim_extractor = replace(
    base_config,
    system_prompt=(
        "You extract decision-relevant claims that a Groundwater Management Area planner would "
        "evaluate, and you mark how confident the document's language is about each claim."
    ),
    temperature=0.0,
    extra={"response_format": {"type": "json_object"}},
)

claim_task = """
Identify claims in the document that could influence a planning decision (pumping limits,
recharge expectations, drawdown targets, data quality, etc.).
For each claim, give it a confidence level based on how the document hedges it:
- "stated"   — given as a measured / reported fact
- "expected" — projection or expectation
- "uncertain" — explicitly flagged as uncertain or unknown
"""

claim_format = """
JSON object with a single key "claims" whose value is a list of objects with fields:
- claim       (string, the decision-relevant statement)
- confidence  (one of: stated, expected, uncertain)
- evidence    (string, the verbatim phrase that supports your label)
"""

claims = json.loads(run_llm(
    claim_extractor,
    context=document_text,
    task=claim_task.strip(),
    format=claim_format.strip(),
))
print(json.dumps(claims, indent=2))

{
  "claims": [
    {
      "claim": "Texas has adequate water to meet its municipal and industrial needs in 1980 and sustain agriculture and other uses.",
      "confidence": "expected",
      "evidence": "This report reveals that through proper development, Texas has adequate water to meet its municipal and industrial needs in 1980 and sustain agriculture and other uses."
    },
    {
      "claim": "Continuous planning and increasingly comprehensive studies will be essential to bring about the development of the State's water resources necessary to meet its needs beyond the year 1980.",
      "confidence": "expected",
      "evidence": "Studies made during the preparation of this report indicate that continuous planning and increasingly comprehensive studies will be essential to bring about the development of the State's water resources necessary to meet its needs beyond the year 1980."
    },
    {
      "claim": "Needs beyond 1980 can be met by proper and continuous planning for o

## 6. Exercise C — Re-run the DFC → MODFLOW mapping on the same document

Reuse `dfc_mapper` from Exercise A, but feed it the `document_text` loaded in section 5. This is the bridge from "demo prompt with a textbook DFC" to "real planning artifact in your CKAN registration."

In [9]:
# Re-run Exercise A's mapping prompt, but with the loaded document as context.
raw = run_llm(
    dfc_mapper,
    context=document_text,
    task=dfc_task,
    format=dfc_format,
)
mapped_from_doc = json.loads(raw)
print(json.dumps(mapped_from_doc, indent=2))

{
  "mappings": [
    {
      "condition": "1980 municipal and industrial water requirements",
      "modflow_package": "WEL",
      "parameter": "pumping rate",
      "sensitivity": "high",
      "uncertainty_notes": "Population growth and industrial demand projections are uncertain"
    },
    {
      "condition": "Groundwater levels in Upper Coastal Areas",
      "modflow_package": "NPF",
      "parameter": "Kx",
      "sensitivity": "medium",
      "uncertainty_notes": "Hydraulic conductivity values are uncertain due to heterogeneity"
    },
    {
      "condition": "Surface water availability in Canadian River Basin",
      "modflow_package": "RCH",
      "parameter": "recharge flux",
      "sensitivity": "high",
      "uncertainty_notes": "Recharge estimates are uncertain due to climate variability"
    },
    {
      "condition": "Irrigation water demands in Rio Grande Basin",
      "modflow_package": "WEL",
      "parameter": "pumping rate",
      "sensitivity": "high",
      "

## 7. Self-scoring rubric (slide 8)

Before you iterate on the prompt, force yourself to pick a level for each dimension. No half-points.

| Dimension | L1 Unsat. | L2 Basic | L3 Solid | L4 Strong | L5 Exceptional |
|---|---|---|---|---|---|
| **DFC understanding** — did it parse the planning target right? | Restates words; misses the target value. | Picks up the number; misses horizon or scope. | Captures number, horizon, and aquifer extent. | Also surfaces ambiguity in the DFC text. | Identifies inconsistencies a planner would flag. |
| **MODFLOW parameter mapping** — right package + parameter for each condition? | Names MODFLOW but no package or parameter. | One correct package; misses the others. | Maps WEL, RCH, K, S to the right conditions. | Distinguishes calibrated vs. forecast inputs. | Names sensitivity hierarchy and trade-offs. |
| **Hydrogeological realism** — would a hydrogeologist sign off? | Confidently wrong; physically inconsistent. | Mostly textbook; ignores site context. | Reasonable for the aquifer; minor errors. | Aware of regional context and key data gaps. | Could be lifted into a TWDB technical memo. |

**Discussion prompts** (last 10 minutes):

- Would you trust this output enough to share with a planner? Why or why not?
- Name one fix to the prompt that would move the lowest dimension up a level — then try it.